# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faress1212/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## My lane: Refresh / Content Opportunity Scoring

I'm choosing this lane because I already ran the starter pipeline in
Week 1-2 and saw a learned model beat a hand-written rule on this exact
problem (Precision@50: 0.740 for the random forest vs 0.240 for the
hand rule). That gap tells me there's real signal in this data that a
simple rule misses — worth spending the next 7 weeks digging into.

In [7]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faress1212/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship


In [8]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])

Rows: 30000 | Columns: 44


## The question: decision, action, cost of a wrong call

**Decision:** Which content pages should a review team look at first
when deciding what to refresh, expand, or leave alone?

**Unit of analysis:** one content page (content_id) per client.

**Who acts on it:** a content reviewer or SEO lead with limited time,
who can only manually review a small number of pages per week.

**Action:** the reviewer opens the top-ranked pages and decides whether
to refresh the content, expand it, or leave it as-is.

**Cost of a wrong recommendation:**
- False positive: reviewer time wasted on a page that wasn't actually
  declining or worth fixing.
- False negative: a real declining page never gets reviewed, and its
  traffic keeps dropping.

**Why ML can help:** a fixed hand-written rule already exists and gets
roughly 1 in 4 of its top picks right. A learned model found nearly 3x
more real signal on the same data, which suggests there are patterns a
simple rule can't capture.

In [9]:
# no code needed here — quick sanity check on the label instead
declining_rate = df["trend_direction"].str.lower().eq("down").mean()
print(f"Share of pages currently in a declining trend: {declining_rate:.3f}")

Share of pages currently in a declining trend: 0.542


## Quick look at the data

Three real numbers from the starter dataset that support this lane:

In [10]:
stale_and_visible = ((df["days_since_last_update"] >= 180) &
                      (df["impressions_90d"] >= 500)).mean()
print(f"1) Share of pages that are stale (180+ days) AND still visible (500+ impressions): {stale_and_visible:.3f}")

declining_with_demand = ((df["trend_direction"].str.lower() == "down") &
                          (df["impressions_90d"] >= 100)).mean()
print(f"2) Share of pages declining with real demand (100+ impressions): {declining_with_demand:.3f}")

print(f"3) Median impressions_90d across all pages: {df['impressions_90d'].median():.0f}")

1) Share of pages that are stale (180+ days) AND still visible (500+ impressions): 0.001
2) Share of pages declining with real demand (100+ impressions): 0.438
3) Median impressions_90d across all pages: 731


## Careful words: what I can and can't claim

**What I can claim:**
- I can show, with client-holdout validation, that a learned ranking
  can outperform a simple hand-written rule on this data.
- I can produce a ranked list of pages as decision-support for a
  human reviewer — not a final verdict.
- My findings are observed and directional, based on this starter
  slice and, later, the warehouse release.

**What I cannot claim:**
- That refreshing a page CAUSES it to recover — that requires a real
  experiment (A/B test or similar), which this data does not give me.
- That my model reflects Google's actual ranking algorithm.
- That results on the 30,000-row starter slice will hold unchanged on
  the full ~79M-row warehouse without re-validating there.

In [11]:
# no additional code needed for this section
print("See markdown above for claims and limits.")

See markdown above for claims and limits.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.